In [17]:

import os
import pickle
import mlflow
import mlflow.keras
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from mlflow.models import infer_signature


In [18]:

tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
if tracking_uri:
    mlflow.set_tracking_uri(tracking_uri)

mlflow.set_experiment("customer-churn-ann")


<Experiment: artifact_location='/home/prasanna/annclassification/mlruns/1', creation_time=1775240918941, experiment_id='1', last_update_time=1775240918941, lifecycle_stage='active', name='customer-churn-ann', tags={}, workspace='default'>

In [19]:

# Load dataset
data = pd.read_csv("Churn_Modelling.csv")
data.head()


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [20]:

# Drop irrelevant columns
data = data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)
data.head()


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [21]:

# Encode Gender
label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(data["Gender"])

# One-hot encode Geography
onehot_encoder_geo = OneHotEncoder()
geo_encoded = onehot_encoder_geo.fit_transform(data[["Geography"]]).toarray()
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(["Geography"])
)

# Combine encoded columns
data = pd.concat([data.drop("Geography", axis=1), geo_encoded_df], axis=1)
data.head()


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [22]:

# Split features and target
X = data.drop("Exited", axis=1)
y = data["Exited"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.shape, X_test_scaled.shape


((8000, 12), (2000, 12))

In [23]:

# Build ANN model
model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

model.summary()


/home/prasanna/annclassification/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [26]:
epochs = 100
batch_size = 32
learning_rate = 0.01

with mlflow.start_run() as run:
    # Log parameters
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("batch_size", batch_size)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("optimizer", "adam")
    mlflow.log_param("loss", "binary_crossentropy")
    mlflow.log_param("hidden_layer_1_units", 64)
    mlflow.log_param("hidden_layer_2_units", 32)

    # Train model
    history = model.fit(
        X_train_scaled,
        y_train,
        validation_data=(X_test_scaled, y_test),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stopping],
        verbose=1
    )

    # Log train/validation metrics
    mlflow.log_metric("train_loss", float(history.history["loss"][-1]))
    mlflow.log_metric("val_loss", float(history.history["val_loss"][-1]))
    mlflow.log_metric("train_accuracy", float(history.history["accuracy"][-1]))
    mlflow.log_metric("val_accuracy", float(history.history["val_accuracy"][-1]))

    # Evaluate on test data
    y_pred_proba = model.predict(X_test_scaled, verbose=0).ravel()
    y_pred = (y_pred_proba >= 0.5).astype(int)

    mlflow.log_metric("test_accuracy", float(accuracy_score(y_test, y_pred)))
    mlflow.log_metric("precision", float(precision_score(y_test, y_pred)))
    mlflow.log_metric("recall", float(recall_score(y_test, y_pred)))
    mlflow.log_metric("f1_score", float(f1_score(y_test, y_pred)))
    mlflow.log_metric("roc_auc", float(roc_auc_score(y_test, y_pred_proba)))

    # Save preprocessors locally and log them
    with open("label_encoder_gender.pkl", "wb") as f:
        pickle.dump(label_encoder_gender, f)

    with open("onehot_encoder_geo.pkl", "wb") as f:
        pickle.dump(onehot_encoder_geo, f)

    with open("scaler.pkl", "wb") as f:
        pickle.dump(scaler, f)

    mlflow.log_artifact("label_encoder_gender.pkl")
    mlflow.log_artifact("onehot_encoder_geo.pkl")
    mlflow.log_artifact("scaler.pkl")

    # Optional local save
    model.save("model.h5")

    # Log model to MLflow
    sample_input = X_train_scaled[:5]
    sample_output = model.predict(sample_input, verbose=0)
    signature = infer_signature(sample_input, sample_output)

    mlflow.keras.log_model(
        model,
        name="model",
        signature=signature
    )

    run_id = run.info.run_id
    print("MLflow Run ID:", run_id)
    print("Model URI:", f"runs:/{run_id}/model")

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8626 - loss: 0.3388 - val_accuracy: 0.8600 - val_loss: 0.3446
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8637 - loss: 0.3365 - val_accuracy: 0.8585 - val_loss: 0.3409
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8662 - loss: 0.3337 - val_accuracy: 0.8575 - val_loss: 0.3458
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8649 - loss: 0.3331 - val_accuracy: 0.8620 - val_loss: 0.3414
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8619 - loss: 0.3307 - val_accuracy: 0.8600 - val_loss: 0.3435
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8662 - loss: 0.3263 - val_accuracy: 0.8605 - val_loss: 0.3439
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8670 - loss: 0.3254 - val_accuracy: 0.8555 - val_loss: 0.3511
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8651 - loss: 0.3254 - val_accu

MLflow Run ID: 3bb1f5cfd4d345c7a1ac8e7cc6d31ffb
Model URI: runs:/3bb1f5cfd4d345c7a1ac8e7cc6d31ffb/model
